1 Setup libraries

In [ ]:
import json
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Set display options for better visibility in the notebook
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Define file paths (this is my path, change to your)
input_file = r'C:\Users\Admin\Documents\ML-cos3049\ML\data\data.json'
output_file = r'C:\Users\Admin\Documents\ML-cos3049\ML\data\data_labeled.json'

2 Get the file path

In [ ]:
# Read data
df = pd.read_json(input_file) 
display(df.head()) # Show the first few rows in the notebook


3 Processing data


In [ ]:
# Processing Data
# 1. Clean the number columns
numeric_columns = ['Transaction amount', 'Account balance', 'Salary (per month)']
for column in numeric_columns:
    df[column] = df[column].astype(str)
    df[column] = df[column].str.replace(',', '', regex=False).str.replace('.', '', regex=False)
    df[column] = pd.to_numeric(df[column], errors='coerce').fillna(0)

# 2. Time processing
if 'Timestamp' in df.columns:
    df['DateTime'] = pd.to_datetime(df['Timestamp'], unit='ms')
    df['Hour'] = df['DateTime'].dt.hour
    df['DayOfWeek'] = df['DateTime'].dt.dayofweek

# 3. Label Encoding
text_columns = ['Transaction Detail', 'Geological', 'Device Use', 'Gender', 'Location', 'Working Status']
df_processed = df.copy()

for column in text_columns:
    if column in df.columns:
        encoder = LabelEncoder()
        new_column_name = column + "_encoded"
        df_processed[new_column_name] = encoder.fit_transform(df[column].astype(str))

df_processed.head()

4 Picking features and Training model

In [ ]:
features = [
    'Transaction amount', 'Account balance', 'Salary (per month)',
    'Hour', 'DayOfWeek',
    'Transaction Detail_encoded', 'Geological_encoded', 
    'Device Use_encoded', 'Location_encoded', 'Working Status_encoded'
]

selected_features = [col for col in features if col in df_processed.columns]
X = df_processed[selected_features]

model = IsolationForest(n_estimators=100, contamination=0.15, random_state=42)
model.fit(X)

In [ ]:
# Predict (-1 is Anomaly, 1 is Normal)
predictions = model.predict(X)

# Assign labels: 1 = Fraud, 0 = Normal
df['is_fraud'] = [1 if p == -1 else 0 for p in predictions]
df['anomaly_score'] = model.score_samples(X)

fraud_count = df['is_fraud'].sum()
print(f"Found {fraud_count} suspicious transactions.")

Show top 5 anomalies

In [ ]:
# Top 5 anomalies
# Filter Fraud transactions and sort by anomaly score
frauds = df[df['is_fraud'] == 1].sort_values('anomaly_score')

columns_to_show = [
    'Transaction ID', 'Transaction amount', 'Transaction Detail', 
    'Location', 'anomaly_score'
]

frauds[columns_to_show].head(5)

5 Visualization


5.1 Anomaly Score Distribution (most important)
    Plot a histogram of anomaly_score. You'll see a bimodal distribution — normal transactions cluster on the right (scores near 0), anomalies cluster on the left (very negative scores).

In [ ]:
import matplotlib.pyplot as plt
plt.hist(df['anomaly_score'], bins=50, color='steelblue')
plt.axvline(df[df['is_fraud']==1]['anomaly_score'].max(), color='red', linestyle='--', label='Fraud threshold')
plt.title('Anomaly Score Distribution')
plt.xlabel('Anomaly Score'); plt.legend(); plt.show()

5.2 Fraud vs Normal Count (Bar Chart)
    Simple but effective — shows how many transactions were flagged.

In [ ]:
df['is_fraud'].value_counts().rename({0:'Normal', 1:'Fraud'}).plot(kind='bar', color=['green','red'])
plt.title('Normal vs Fraud Count')


5.3 Transaction Amount vs Account Balance (Scatter Plot)
    Color points by is_fraud to see if anomalies cluster in specific regions (e.g., huge amounts with low balance).

In [ ]:
plt.scatter(df['Transaction amount'], df['Account balance'], c=df['is_fraud'], cmap='coolwarm', alpha=0.5)
plt.xlabel('Transaction Amount'); plt.ylabel('Account Balance')
plt.title('Transaction Amount vs Account Balance')


5.4 Anomaly Heatmap by Hour and Day of Week
    Shows when fraud happens — useful since code extracts Hour and DayOfWeek.

In [ ]:
import seaborn as sns
pivot = df[df['is_fraud']==1].groupby(['DayOfWeek','Hour']).size().unstack(fill_value=0)
sns.heatmap(pivot, cmap='Reds')
plt.title('Fraud Heatmap: Hour vs Day of Week')
